# Example queries: `timeseries` (resstock_oedi)

Auto-generated from `tests/query_snapshots/timeseries.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading resstock_2024_amy2018_release_2 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `ts_monthly_electricity_by_state`

Monthly electricity grouped by state (CO only), upgrade 0 baseline.


In [3]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='month',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01,9425,2.377943e+06,2976,2.529477e+09
1,CO,2018-02-01,9425,2.377943e+06,2688,2.335684e+09
2,CO,2018-03-01,9425,2.377943e+06,2976,1.849398e+09
3,CO,2018-04-01,9425,2.377943e+06,2880,1.588177e+09
4,CO,2018-05-01,9425,2.377943e+06,2976,1.477754e+09


## `ts_monthly_two_fuels_by_building_type`

Monthly electricity + natural gas grouped by building type and time, CO only.


In [4]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption', 'out.natural_gas.total.energy_consumption'],
    annual_only=False,
    timestamp_grouping_func='month',
    group_by=['geometry_building_type_recs', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption,natural_gas.total.energy_consumption
0,Mobile Home,2018-01-01,391,98649.940744,2976,8.204234e+07,2.164715e+08
1,Mobile Home,2018-02-01,391,98649.940744,2688,7.427992e+07,1.981446e+08
2,Mobile Home,2018-03-01,391,98649.940744,2976,6.740720e+07,1.283149e+08
3,Mobile Home,2018-04-01,391,98649.940744,2880,6.014728e+07,9.406706e+07
4,Mobile Home,2018-05-01,391,98649.940744,2976,5.758652e+07,4.205395e+07


## `ts_year_collapse_electricity`

Year-collapsed timeseries electricity, grouped by building type, CO only.


In [5]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    timestamp_grouping_func='year',
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,Mobile Home,391,9.864994e+04,35040,7.898218e+08
1,Multi-Family with 2 - 4 Units,469,1.183295e+05,35040,8.547433e+08
2,Multi-Family with 5+ Units,2001,5.048556e+05,35040,3.249322e+09
3,Single-Family Attached,664,1.675283e+05,35040,1.345610e+09
4,Single-Family Detached,5900,1.488580e+06,35040,1.635436e+10


## `ts_15min_raw_electricity_by_state`

Raw 15-min cadence timeseries (no timestamp_grouping_func), CO only. Exercises the un-aggregated time-axis code path that example notebooks rely on.


In [6]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,electricity.total.energy_consumption
0,CO,2018-01-01 00:15:00,9425,2.377943e+06,1.347721e+06
1,CO,2018-01-01 00:30:00,9425,2.377943e+06,1.360706e+06
2,CO,2018-01-01 00:45:00,9425,2.377943e+06,1.333647e+06
3,CO,2018-01-01 01:00:00,9425,2.377943e+06,1.321425e+06
4,CO,2018-01-01 01:15:00,9425,2.377943e+06,1.227585e+06


## `ts_daily_electricity_by_state`

Daily-grouped timeseries electricity, CO only. Exercises date_trunc('day', ...) plus the rows_per_sample / distinct-bs-keys grouping_metrics branch.


In [7]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='day',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01,9425,2.377943e+06,96,1.126994e+08
1,CO,2018-01-02,9425,2.377943e+06,96,9.861118e+07
2,CO,2018-01-03,9425,2.377943e+06,96,8.659704e+07
3,CO,2018-01-04,9425,2.377943e+06,96,8.034337e+07
4,CO,2018-01-05,9425,2.377943e+06,96,7.959871e+07


## `ts_hourly_electricity_by_building_type`

Hourly-grouped timeseries electricity, CO + single building type. Exercises date_trunc('hour', ...) — the smallest grouping_func granularity.


In [8]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='hour',
    group_by=['geometry_building_type_recs', 'time'],
    restrict=[('state', ['CO']), ('geometry_building_type_recs', ['Mobile Home'])],
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,Mobile Home,2018-01-01 00:00:00,391,98649.940744,4,162605.378542
1,Mobile Home,2018-01-01 01:00:00,391,98649.940744,4,134183.598939
2,Mobile Home,2018-01-01 02:00:00,391,98649.940744,4,113917.974411
3,Mobile Home,2018-01-01 03:00:00,391,98649.940744,4,112954.182151
4,Mobile Home,2018-01-01 04:00:00,391,98649.940744,4,107858.193652


## `ts_monthly_multi_attribute_group_by`

Monthly TS electricity grouped by [building_type, state, time] — multi-attribute group_by combined with the time axis. Every other TS entry uses single-attribute + time; this pins the SQL shape for the more general case.


In [9]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='month',
    group_by=['geometry_building_type_recs', 'state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,Mobile Home,CO,2018-01-01,391,98649.940744,2976,8.204234e+07
1,Mobile Home,CO,2018-02-01,391,98649.940744,2688,7.427992e+07
2,Mobile Home,CO,2018-03-01,391,98649.940744,2976,6.740720e+07
3,Mobile Home,CO,2018-04-01,391,98649.940744,2880,6.014728e+07
4,Mobile Home,CO,2018-05-01,391,98649.940744,2976,5.758652e+07


## `ts_hourly_electricity_by_state`

Hourly TS electricity grouped by state, CO only. Restrict and group_by deliberately match ts_daily_electricity_by_state and ts_monthly_electricity_by_state so the hourly→daily→monthly sum invariants align.


In [10]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='hour',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01 00:00:00,9425,2.377943e+06,4,5.363499e+06
1,CO,2018-01-01 01:00:00,9425,2.377943e+06,4,4.908810e+06
2,CO,2018-01-01 02:00:00,9425,2.377943e+06,4,4.082339e+06
3,CO,2018-01-01 03:00:00,9425,2.377943e+06,4,4.162318e+06
4,CO,2018-01-01 04:00:00,9425,2.377943e+06,4,4.251390e+06
